In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, PathPatch
from matplotlib.path import Path

# =========================
# Nature-style Overview Box Diagram v2
# - Fix text overlaps by re-layout + shorter labels
# - Less cramped boxes, cleaner typography
# - Curly brace on RIGHT of 3 hindcast design boxes
# - Compact hindcast-design descriptions
# - No "extendable slots"
# =========================

fig, ax = plt.subplots(figsize=(14.5, 7.6))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")


def add_box(x, y, w, h, text, fs=12, fc="#f3f3f3", ec="black", lw=1.7, rs=0.03):
    b = FancyBboxPatch(
        (x, y), w, h,
        boxstyle=f"round,pad=0.018,rounding_size={rs}",
        facecolor=fc, edgecolor=ec, linewidth=lw, joinstyle="round"
    )
    ax.add_patch(b)
    ax.text(x + w/2, y + h/2, text,
            ha="center", va="center", fontsize=fs, wrap=True)
    return b


def add_arrow(p1, p2, text=None, fs=9.5, rad=0.0, text_offset=(0, 0.0)):
    arr = FancyArrowPatch(
        p1, p2,
        arrowstyle='-|>',
        mutation_scale=17,
        linewidth=1.5,
        color="black",
        connectionstyle=f"arc3,rad={rad}"
    )
    ax.add_patch(arr)
    if text:
        mx = (p1[0] + p2[0]) / 2 + text_offset[0]
        my = (p1[1] + p2[1]) / 2 + text_offset[1]
        ax.text(mx, my, text, ha="center", va="center", fontsize=fs)
    return arr


def add_curly_brace_right(x, y0, y1, width=0.02, lw=1.5):
    """
    Vertical curly brace on the RIGHT side spanning y0..y1 at x.
    """
    mid = (y0 + y1) / 2
    verts = [
        (x, y1),
        (x + width, y1), (x + width, mid + 0.045), (x, mid + 0.02),
        (x - width, mid), (x + width, mid - 0.045), (x, mid - 0.02),
        (x + width, y0), (x + width, y0), (x, y0),
    ]
    codes = [
        Path.MOVETO,
        Path.CURVE4, Path.CURVE4, Path.CURVE4,
        Path.CURVE4, Path.CURVE4, Path.CURVE4,
        Path.CURVE4, Path.CURVE4, Path.CURVE4
    ]
    path = Path(verts, codes)
    patch = PathPatch(path, fill=False, linewidth=lw, color="black")
    ax.add_patch(patch)
    return patch


# --- Top forcing/IC box ---
add_box(0.33, 0.90, 0.34, 0.07,
        "Forcing & Initial Conditions", fs=14, fc="#ffffff", lw=2.0, rs=0.04)

# --- Main three boxes (more spacing + slightly taller) ---
longrun = add_box(
    0.04, 0.56, 0.26, 0.28,
    "Long-run\n(0001–0024, ~23 yrs)\n\n"
    "Interactive:\nDynamics ↔ Chemistry ↔ Radiation\nO₃ feedback ON",
    fs=11.2
)

clim = add_box(
    0.36, 0.56, 0.26, 0.28,
    "Climatology\n(200 yrs, 101–300)\n\n",
    fs=11.0
)

hind = add_box(
    0.68, 0.50, 0.28, 0.34,
    "Hindcast System\n(re-simulate extremes)\n\n"
    "Target years:\n0003, 0008, 0013, 0014, 0019",
    fs=11.2
)

# --- arrows from forcing ---
add_arrow((0.50, 0.90), (0.17, 0.85))
add_arrow((0.50, 0.90), (0.49, 0.85))
add_arrow((0.50, 0.90), (0.82, 0.85))

# --- Longrun -> Hindcast: label BELOW arrow to avoid overlap ---
add_arrow((0.30, 0.69), (0.68, 0.69),
          text="Identify extreme O₃-loss years", fs=9.5, text_offset=(0, -0.035))

# --- Climatology -> Hindcast: short label, slightly lower ---
add_arrow((0.62, 0.64), (0.68, 0.62),
          text="O₃ clim for nochem", fs=9.3, text_offset=(0, -0.03))

# --- Hindcast design sub-boxes (compact text) ---
sub_x = 0.71
sub_w = 0.24
sub_h = 0.085
sub_gap = 0.018
sub_y_top = 0.33

b1 = add_box(
    sub_x, sub_y_top, sub_w, sub_h,
    "Perturbations\nsmall(10⁻⁷?) / large(10⁻⁶?) / Q(H₂O?, 0008-MAR)",
    fs=9.3
)
b2 = add_box(
    sub_x, sub_y_top - (sub_h + sub_gap), sub_w, sub_h,
    "Start months\n0008: JAN–APR\nothers: FEB–MAR",
    fs=9.5
)
b3 = add_box(
    sub_x, sub_y_top - 2*(sub_h + sub_gap), sub_w, sub_h,
    "O₃ coupling\nchem(int): FB ON\nnochem: uses O₃ clim (?)",
    fs=9.3
)

# arrows from hindcast to stack
add_arrow((0.82, 0.50), (0.82, sub_y_top + sub_h + 0.008))
add_arrow((0.82, sub_y_top), (0.82, sub_y_top - sub_gap))
add_arrow((0.82, sub_y_top - (sub_h + sub_gap)),
          (0.82, sub_y_top - (sub_h + sub_gap) - sub_gap))

# --- Curly brace on RIGHT of three design boxes ---
brace_x = sub_x + sub_w + 0.015
brace_y0 = sub_y_top - 2*(sub_h + sub_gap)
brace_y1 = sub_y_top + sub_h
add_curly_brace_right(brace_x, brace_y0, brace_y1, width=0.018)

ax.text(brace_x + 0.03, (brace_y0 + brace_y1)/2,
        "Hindcast\ndesign", ha="left", va="center", fontsize=10.2)

# --- Hindcast-year inventory (shortened lines, bigger box) ---
years_box = add_box(
    0.05, 0.08, 0.60, 0.30,
    "Hindcast experiments by year:\n\n"
    "0008: small(JAN–APR) + nochem(FEB/MAR) + large(FEB/MAR/APR) + Q(MAR)\n"
    "0013/0014/0019: small(FEB/MAR) + nochem(FEB/MAR)\n"
    "0003: small(FEB/MAR)",
    fs=10.4
)

# connector from hindcast to year box (clean arc)
add_arrow((0.68, 0.52), (0.64, 0.38), rad=-0.18)

# --- Save (your required style) ---
out_path = "/home/weiji/test_code/plots/New_weiji_general_why0008important/nature_style_overview_v1.png"
plt.show()
plt.savefig(out_path, dpi=220, bbox_inches="tight")
print("Saved:", out_path)
